# Temperature summary (V5): calibrated barrier heights + FN summary

Aggregates the per-temperature `fit_summary_{T}K.csv` files written by the `Spectroscopy and FN analysis_{T}K.ipynb` notebooks. Each per-T CSV holds the Method-2 binned $\Phi_\mathrm{AFM}$, $\Phi_\mathrm{FM}$, $\Delta_\mathrm{ex}$ for both axes, the below-saturation $\cos(\theta/2)$ and $\sin^2(\theta/2)$ fit parameters, and the AFM-anchored transition-voltage calibration ($c = V_\mathrm{peak}^\mathrm{AFM}/V_T^\mathrm{AFM}$, $V_T^\mathrm{AFM}$, `calib_ok`).

**Calibrated window:** the absolute $\Phi(T)$ plots use only $T \leq 80$ K, where a clean AFM FN minimum resolves. At 90 and 100 K the field-emission regime is lost, so those points carry no absolute calibration and are reported as relative peak splitting only (Section 7c).

## 1. Imports and paths

In [ ]:
from scripts.utils import setup_notebook, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

V4_ROOT = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'barrier_vs_canting'
C_DIR   = V4_ROOT / 'c_scans'
OUT_DIR = V4_ROOT / 'summary'
OUT_DIR.mkdir(parents=True, exist_ok=True)

T_MAX_PLOT = 80  # K; plots are restricted to T <= this value.

print('per-T CSVs:', C_DIR)
print('summary out:', OUT_DIR)
print('plot T cutoff:', T_MAX_PLOT, 'K')


## 2. Load per-temperature `fit_summary` CSVs

Each per-T notebook writes a one-row CSV. We concatenate them into one DataFrame sorted by temperature, then build a `plot_df` view restricted to $T \leq T_{\rm MAX}$.

In [ ]:
import re

rows = []
for fp in sorted(C_DIR.glob('fit_summary_*K.csv')):
    m = re.match(r'fit_summary_(\d+)K\.csv', fp.name)
    if not m:
        continue
    df = pd.read_csv(fp)
    if len(df) != 1:
        print(f'  skip {fp.name}: expected 1 row, got {len(df)}')
        continue
    rows.append(df.iloc[0])

summary = pd.DataFrame(rows).sort_values('temperature_K').reset_index(drop=True)
plot_df = summary[summary['temperature_K'] <= T_MAX_PLOT].reset_index(drop=True)
print(f'Loaded {len(summary)} temperatures: {summary["temperature_K"].tolist()}')
print(f'Plotted ({T_MAX_PLOT} K cutoff): {plot_df["temperature_K"].tolist()}')
summary


## 3. $\Phi_{\rm AFM}(T)$ and $\Phi_{\rm FM}(T)$ — Method 2 binned endpoints

Inverse-variance weighted mean of binned $\Phi$ in the AFM ($|H| < H_{\rm AFM}^{\max}$) and FM ($|H| > H_{\rm FM}^{\min,\rm eff}$) windows, for both c-axis ($H_\mathrm{z}$) and b-axis ($H_\mathrm{y}$).

In [ ]:
T = plot_df['temperature_K'].values

fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
ax.errorbar(T, plot_df['Phi_AFM_c_m2_meV'], yerr=plot_df['Phi_AFM_c_m2_err_meV'],
            fmt='o', color=OKABE_ITO_CYCLE[1], capsize=2,
            label=r'$\Phi_{\rm AFM}$, c-axis')
ax.errorbar(T, plot_df['Phi_AFM_b_m2_meV'], yerr=plot_df['Phi_AFM_b_m2_err_meV'],
            fmt='s', color=OKABE_ITO_CYCLE[1], capsize=2, mfc='white',
            label=r'$\Phi_{\rm AFM}$, b-axis')
ax.errorbar(T, plot_df['Phi_FM_c_m2_meV'], yerr=plot_df['Phi_FM_c_m2_err_meV'],
            fmt='o', color=OKABE_ITO_CYCLE[5], capsize=2,
            label=r'$\Phi_{\rm FM}$, c-axis')
ax.errorbar(T, plot_df['Phi_FM_b_m2_meV'], yerr=plot_df['Phi_FM_b_m2_err_meV'],
            fmt='s', color=OKABE_ITO_CYCLE[5], capsize=2, mfc='white',
            label=r'$\Phi_{\rm FM}$, b-axis')
ax.set_xlabel(r'$T$ (K)')
ax.set_ylabel(r'$\Phi$ (meV)')
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(OUT_DIR / 'Phi_AFM_Phi_FM_vs_T.png', dpi=300)
plt.show()

print('Phi_AFM and Phi_FM (meV)')
for _, r in plot_df.iterrows():
    print(f'  T={int(r["temperature_K"]):>4d} K  '
          f'AFM c={r["Phi_AFM_c_m2_meV"]:7.2f}+/-{r["Phi_AFM_c_m2_err_meV"]:5.2f}  '
          f'AFM b={r["Phi_AFM_b_m2_meV"]:7.2f}+/-{r["Phi_AFM_b_m2_err_meV"]:5.2f}  '
          f'FM c={r["Phi_FM_c_m2_meV"]:7.2f}+/-{r["Phi_FM_c_m2_err_meV"]:5.2f}  '
          f'FM b={r["Phi_FM_b_m2_meV"]:7.2f}+/-{r["Phi_FM_b_m2_err_meV"]:5.2f}')


## 4. $\Delta_{\rm ex}(T)$ — exchange-driven gap

$\Delta_{\rm ex} = \Phi_{\rm AFM} - \Phi_{\rm FM}$ from Method 2 binned endpoints on both axes.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
ax.errorbar(T, plot_df['Delta_ex_c_m2_meV'], yerr=plot_df['Delta_ex_c_m2_err_meV'],
            fmt='o', color=OKABE_ITO_CYCLE[2], capsize=2, label='c-axis')
ax.errorbar(T, plot_df['Delta_ex_b_m2_meV'], yerr=plot_df['Delta_ex_b_m2_err_meV'],
            fmt='s', color=OKABE_ITO_CYCLE[2], capsize=2, mfc='white', label='b-axis')
ax.axhline(0.0, color='0.5', lw=0.8, ls='--')
ax.set_xlabel(r'$T$ (K)')
ax.set_ylabel(r'$\Delta_{\rm ex}$ (meV)')
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(OUT_DIR / 'Delta_ex_vs_T.png', dpi=300)
plt.show()

print('Delta_ex (meV)')
for _, r in plot_df.iterrows():
    print(f'  T={int(r["temperature_K"]):>4d} K  '
          f'c={r["Delta_ex_c_m2_meV"]:7.2f}+/-{r["Delta_ex_c_m2_err_meV"]:5.2f}  '
          f'b={r["Delta_ex_b_m2_meV"]:7.2f}+/-{r["Delta_ex_b_m2_err_meV"]:5.2f}')


## 5. Fit quality: $\chi^2_{\rm red}$ vs $T$

Reduced chi-squared for the c-axis below-saturation fits — linear $\Phi=a+b\,|H_\mathrm{z}|$ and $\sin^2(\theta/2)$ $\Phi=a+b\,[1-(H_\mathrm{z}/H_{\rm sat})^2]$ — as a function of temperature. Closer to 1 means the model is consistent with the per-bin error bars; large values indicate the data depart from the assumed functional form.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
ax.plot(T, plot_df['lin_chi2_red'], 'o-', color=OKABE_ITO_CYCLE[2], label='Linear fit')
ax.plot(T, plot_df['sin2_chi2_red'], 's-', color=OKABE_ITO_CYCLE[4],
        label=r'$\sin^2(\theta/2)$ fit')
ax.axhline(1.0, color='0.5', lw=0.8, ls='--')
ax.set_xlabel(r'$T$ (K)')
ax.set_ylabel(r'$\chi^2_{\rm red}$')
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(OUT_DIR / 'chi2_red_vs_T.png', dpi=300)
plt.show()

print('chi^2_red    linear     sin^2(theta/2)')
for _, r in plot_df.iterrows():
    print(f'  T={int(r["temperature_K"]):>4d} K  '
          f'lin={r["lin_chi2_red"]:6.2f} (chi^2={r["lin_chi2"]:7.2f}, dof={int(r["lin_dof"])})  '
          f'sin2={r["sin2_chi2_red"]:6.2f} (chi^2={r["sin2_chi2"]:7.2f}, dof={int(r["sin2_dof"])})')


## 5b. Per-temperature fit-form comparison (cos vs sin$^2$, supplementary)

Direct visual + $\chi^2$ comparison of the two candidate canting laws for the
below-saturation c-axis barrier: the coherent-hybridization law
$\Phi = a + b\cos(\theta/2)$ (linear in $\cos(\theta/2)=|H_\mathrm{z}|/H_\mathrm{sat}^\mathrm{c}$)
versus the classical-projection law $\Phi = a + b\sin^2(\theta/2)$. Both fits are
inverse-variance weighted with the Method-2 binned errors, exactly as in each
`Spectroscopy and FN analysis_{T}K` notebook; the $\chi^2$ shown here are
recomputed from the saved binned CSVs so they match the per-T notebooks.

Each row is one temperature: the linear $\cos(\theta/2)$ fit (left) consistently
reaches a lower $\chi^2$ than $\sin^2(\theta/2)$ (right). Figure 1 collects
$T \leq 70$ K; Figure 2 collects 80-100 K (where, above $\sim 80$ K, $\Phi$ is
uncalibrated, so only the *functional form* is compared, not absolute values).

In [ ]:
# ===== Per-temperature cos(theta/2) vs sin^2(theta/2) fit-form comparison =====
def weighted_linear_fit(x, y, yerr):
    '''Inverse-variance weighted y = a + b*x; returns a, b, chi2, chi2_red, dof, n.

    Identical recipe to the per-T 'Spectroscopy and FN analysis' notebooks, so
    chi^2 reproduces those notebooks exactly.'''
    x = np.asarray(x, float); y = np.asarray(y, float); yerr = np.asarray(yerr, float)
    pos = yerr > 0
    safe = np.where(pos, yerr, np.median(yerr[pos]) if pos.any() else 1.0)
    w = 1.0 / safe**2
    S, Sx, Sxx = w.sum(), (w * x).sum(), (w * x * x).sum()
    Sy, Sxy = (w * y).sum(), (w * x * y).sum()
    det = S * Sxx - Sx * Sx
    a = (Sxx * Sy - Sx * Sxy) / det
    b = (S * Sxy - Sx * Sy) / det
    resid = y - (a + b * x)
    chi2 = float(np.sum(w * resid ** 2))
    dof = max(int(x.size - 2), 1)
    return dict(a=float(a), b=float(b), chi2=chi2, chi2_red=chi2 / dof,
                dof=dof, n=int(x.size))


def _load_below_sat(T):
    '''Binned below-saturation Phi(H_z) for temperature T (c-axis).'''
    bdf = pd.read_csv(C_DIR / f'Phi_vs_Hz_binned_{T}K.csv')
    Hsat = float(summary.loc[summary['temperature_K'] == T, 'H_sat_c_T'].iloc[0])
    m = (bdf['abs_H'].values < Hsat) & np.isfinite(bdf['Phi_err_m2_eV'].values)
    sub = bdf.loc[m]
    cos_half = sub['abs_H'].values / Hsat
    sin2_half = 1.0 - (sub['abs_H'].values / Hsat) ** 2
    return cos_half, sin2_half, sub['Phi_eV'].values, sub['Phi_err_m2_eV'].values


def fit_comparison_grid(temps, out_name):
    '''Side-by-side cos(theta/2) | sin^2(theta/2) fit, one row per temperature,
    chi^2 in each panel title. Saves OUT_DIR/out_name; prints the chi^2 table.'''
    nrows = len(temps)
    fig, axes = plt.subplots(nrows, 2, figsize=(10, 3.2 * nrows), dpi=150,
                             squeeze=False)
    rows = []
    for i, T in enumerate(temps):
        cos_half, sin2_half, y, yerr = _load_below_sat(T)
        fit_cos = weighted_linear_fit(cos_half, y, yerr)
        fit_sin2 = weighted_linear_fit(sin2_half, y, yerr)
        rows.append((T, fit_cos['chi2'], fit_cos['chi2_red'],
                     fit_sin2['chi2'], fit_sin2['chi2_red'], fit_cos['dof']))
        for ax, x, fit, xlab, chi_ha in [
            (axes[i, 0], cos_half, fit_cos,
             r'$\cos(\theta/2) = |H_\mathrm{z}|/H_\mathrm{sat}^\mathrm{c}$', 'right'),
            (axes[i, 1], sin2_half, fit_sin2,
             r'$\sin^2(\theta/2) = 1-(|H_\mathrm{z}|/H_\mathrm{sat}^\mathrm{c})^2$', 'left')]:
            ax.errorbar(x, 1000 * y, yerr=1000 * yerr, fmt='o', ms=4,
                        color=OKABE_ITO_CYCLE[2], ecolor=OKABE_ITO_CYCLE[2],
                        elinewidth=0.8, capsize=2, label='Data')
            xx = np.linspace(float(np.min(x)), float(np.max(x)), 100)
            ax.plot(xx, 1000 * (fit['a'] + fit['b'] * xx), '-', color='black',
                    lw=1.3, label='Linear fit')
            # chi^2 annotation inside the panel: top-right on the cos (left)
            # panel, top-left on the sin^2 (right) panel.
            chi_txt = (rf'$\chi^2 = {fit["chi2"]:.1f}$' + '\n'
                       + rf'$\chi^2_\mathrm{{red}} = {fit["chi2_red"]:.2f}$')
            chi_x = 0.96 if chi_ha == 'right' else 0.04
            ax.text(chi_x, 0.96, chi_txt, transform=ax.transAxes,
                    ha=chi_ha, va='top')
            # x-labels on the bottom row only (matches the FN-grid cell).
            if i == nrows - 1:
                ax.set_xlabel(xlab)
        axes[i, 0].set_ylabel(r'$\Phi$ (meV)')
        # temperature: lower-left of the left (cos) panel only.
        axes[i, 0].text(0.04, 0.04, f'{T} K', transform=axes[i, 0].transAxes,
                        ha='left', va='bottom', fontweight='bold')
    # legend on the first right panel's empty lower-right corner.
    axes[0, 1].legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(OUT_DIR / out_name, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'{"T":>5}  {"chi2_cos":>9} {"chi2r_cos":>10}  '
          f'{"chi2_sin2":>9} {"chi2r_sin2":>11}  dof')
    for T, c2c, c2rc, c2s, c2rs, dof in rows:
        print(f'{T:>4}K  {c2c:9.1f} {c2rc:10.2f}  {c2s:9.1f} {c2rs:11.2f}  {dof}')
    return rows

In [ ]:
# T <= 70 K (calibrated window): 6 temperatures, one row each.
_ = fit_comparison_grid([20, 30, 40, 50, 60], 'Phi_fit_comparison_grid_le70K.png')

In [ ]:
# 80-100 K: 80 K calibrated; 90 and 100 K uncalibrated (form-only).
_ = fit_comparison_grid([70, 80, 90, 100], 'Phi_fit_comparison_grid_80_100K.png')

## 7. FN transition voltage and calibration factor vs $T$

### 7a. $eV_T^\mathrm{AFM} = \Phi_\mathrm{AFM}(T)$

The AFM transition voltage measured from the FN minimum, which anchors the calibration ($\Phi_\mathrm{AFM} = eV_T^\mathrm{AFM}$ by construction). Only temperatures with a resolved minimum appear.

In [ ]:
Tv = summary['temperature_K'].values
m_c = np.isfinite(summary['V_T_afm_c_meV'].values)
m_b = np.isfinite(summary['V_T_afm_b_meV'].values)

fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
ax.plot(Tv[m_c], summary['V_T_afm_c_meV'].values[m_c], 'o-',
        color=OKABE_ITO_CYCLE[1], label='c-axis')
ax.plot(Tv[m_b], summary['V_T_afm_b_meV'].values[m_b], 's-',
        color=OKABE_ITO_CYCLE[5], mfc='white', label='b-axis')
ax.set_xlabel(r'$T$ (K)')
ax.set_ylabel(r'$eV_T^\mathrm{AFM} = \Phi_\mathrm{AFM}$ (meV)')
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(OUT_DIR / 'V_T_afm_vs_T.png', dpi=300)
plt.show()

print('V_T^AFM = Phi_AFM (meV)')
for _, r in summary.iterrows():
    if np.isfinite(r['V_T_afm_c_meV']):
        print(f"  T={int(r['temperature_K']):>4d} K  c={r['V_T_afm_c_meV']:6.0f}  "
              f"b={r['V_T_afm_b_meV']:6.0f}")


### 7b. Calibration factor $c(T) = V_\mathrm{peak}^\mathrm{AFM}/V_T^\mathrm{AFM}$

The per-axis factor stays in a narrow band across the calibrated window, so the conversion $\Phi = eV_\mathrm{peak}/c$ is stable.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
ax.plot(Tv[m_c], summary['c_calib_c'].values[m_c], 'o-',
        color=OKABE_ITO_CYCLE[2], label='c-axis')
ax.plot(Tv[m_b], summary['c_calib_b'].values[m_b], 's-',
        color=OKABE_ITO_CYCLE[4], mfc='white', label='b-axis')
ax.set_xlabel(r'$T$ (K)')
ax.set_ylabel(r'$c = V_\mathrm{peak}^\mathrm{AFM}/V_T^\mathrm{AFM}$')
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(OUT_DIR / 'calibration_factor_vs_T.png', dpi=300)
plt.show()

print('calibration factor c')
for _, r in summary.iterrows():
    if np.isfinite(r['c_calib_c']):
        print(f"  T={int(r['temperature_K']):>4d} K  c-axis={r['c_calib_c']:.3f}  "
              f"b-axis={r['c_calib_b']:.3f}")


### 7c. Uncalibrated high-$T$ regime ($T > 80$ K)

At 90 and 100 K the AFM FN minimum does not resolve, so no absolute barrier height is claimed. Only the relative conductance-peak splitting $\Delta V_\mathrm{peak} = V_\mathrm{peak}^\mathrm{AFM} - V_\mathrm{peak}^\mathrm{FM}$ is reported (in mV of bias, not energy).

In [ ]:
hi = summary[~summary['calib_ok_c'].astype(bool)]
if len(hi):
    print('Uncalibrated regime (no AFM FN minimum) -- relative peak splitting only:')
    for _, r in hi.iterrows():
        dvp_c = r['Delta_ex_c_m2_meV']   # = raw dV_peak when calib_ok is False
        dvp_b = r['Delta_ex_b_m2_meV']
        print(f"  T={int(r['temperature_K']):>4d} K:  dV_peak(c) = {dvp_c:6.0f} mV,  "
              f"dV_peak(b) = {dvp_b:6.0f} mV   (V_T not resolved)")
else:
    print('All loaded temperatures calibrated (AFM V_T resolved).')


## 8. Field-resolved Fowler-Nordheim plots at all temperatures

$\ln(|I|/V^2)$ vs $1/V$ for $|H_\mathrm{z}|$ bins (AFM $\to$ canted $\to$ FM, coolwarm) at each temperature. The dashed vertical marks $1/V_T^\mathrm{AFM}$ where the minimum resolves. The FN minimum is clean at low $T$ and is progressively lost above $\sim 80$ K, which is exactly why 90 and 100 K cannot be calibrated.

In [ ]:
from scripts.IV_Hscan_gaussian import load_dataframe
from scripts import fn_analysis as fn

DATA_DIR = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'dataframes' / 'c_scans'
T_all = summary['temperature_K'].astype(int).tolist()

ncol = 3
nrow = int(np.ceil(len(T_all) / ncol))

# Custom y-limits per row — adjust each tuple as needed
ylims_per_row = [(-17, -13), (-15, -12), (-14, -10.5)] * nrow

fig, axes = plt.subplots(nrow, ncol, figsize=(4.2*ncol, 3.4*nrow), dpi=200,
                         sharex=True, sharey='row', squeeze=False)
cmap = plt.cm.coolwarm
for idx, (ax, T) in enumerate(zip(axes.ravel(), T_all)):
    row = idx // ncol
    try:
        df = load_dataframe(DATA_DIR / f'IV_gaussian_{T}K.pkl')
    except FileNotFoundError:
        ax.axis('off')
        continue
    Hsat = float(summary.loc[summary['temperature_K'] == T, 'H_sat_c_T'].iloc[0])
    centers = list(np.round(np.linspace(0.0, round(Hsat, 2), 8), 2))
    cnorm = plt.Normalize(vmin=min(centers), vmax=max(centers))
    for rec in fn.field_binned_fn(df, centers, bin_hw=0.10):
        if rec['x'] is None:
            continue
        ax.plot(rec['x'], rec['y'], '-', color=cmap(cnorm(rec['H_center'])), lw=1.1)
    vt = summary.loc[summary['temperature_K'] == T, 'V_T_afm_c_meV'].iloc[0]
    ax.text(0.96, 0.96, f'{T} K', transform=ax.transAxes, ha='right', va='top')
    ax.set_xlim(0, 20)
    ax.set_ylim(*ylims_per_row[row])
for ax in axes.ravel()[len(T_all):]:
    ax.axis('off')
for ax in axes[-1, :]:
    ax.set_xlabel(r'$1/V$ (V$^{-1}$)')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$\ln(|I|/V^2)$')
fig.tight_layout()
fig.savefig(OUT_DIR / 'FN_field_resolved_grid_all_T.png', dpi=200)
plt.show()


## 8b. Field-resolved Fowler-Nordheim plots, b-axis ($H_\mathrm{y}$)

The b-axis (easy, in-plane) analogue of Section 8. $\ln(|I|/V^2)$ vs $1/V$ for
$|H_\mathrm{y}|$ bins (AFM $\to$ spin-flip $\to$ FM, coolwarm) at each
temperature; the dashed vertical marks $1/V_T^\mathrm{AFM}$ (b-axis) where the
minimum resolves. The b-axis spin-flip and saturation sit at much smaller fields
($\sim 0.3$ T) than the c-axis, so the bins span 0-0.6 T. As for the c-axis, the
FN minimum is clean at low $T$ and is lost above $\sim 80$ K (no dashed line at
90 and 100 K, where $V_T^\mathrm{AFM}$ is unresolved).

In [ ]:
# --- b-axis (H_y) analogue of Section 8: field-resolved FN at all T. ---
# Reuses load_dataframe, fn, T_all defined in the Section 8 cell above.
DATA_DIR_B = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'dataframes' / 'b_scans'
B_CENTERS = list(np.round(np.linspace(0.0, 0.60, 8), 2))  # T; b-axis spans ~0-0.6 T

ncol = 3
nrow = int(np.ceil(len(T_all) / ncol))

# Custom y-limits per row -- adjust each tuple as needed.
ylims_per_row = [(-17, -13), (-15, -12), (-14, -10.5)] * nrow

# sharey='row' so each row can carry its own y-limits (inner columns still
# share, hiding redundant y-tick labels).
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.4 * nrow), dpi=200,
                         sharex=True, sharey='row', squeeze=False)
cmap = plt.cm.coolwarm
for ax, T in zip(axes.ravel(), T_all):
    try:
        df = load_dataframe(DATA_DIR_B / f'IV_gaussian_{T}K.pkl')
    except FileNotFoundError:
        ax.axis('off')
        continue
    cnorm = plt.Normalize(vmin=min(B_CENTERS), vmax=max(B_CENTERS))
    for rec in fn.field_binned_fn(df, B_CENTERS, bin_hw=0.04):
        if rec['x'] is None:
            continue
        ax.plot(rec['x'], rec['y'], '-', color=cmap(cnorm(rec['H_center'])), lw=1.1)
    vt = summary.loc[summary['temperature_K'] == T, 'V_T_afm_b_meV'].iloc[0]
    if np.isfinite(vt):
        ax.axvline(1000.0 / vt, color='0.3', ls='--', lw=0.9, alpha=0.7)
    ax.text(0.96, 0.96, f'{T} K', transform=ax.transAxes, ha='right', va='top')
    ax.set_xlim(0, 20)
for ax in axes.ravel()[len(T_all):]:
    ax.axis('off')
# Per-row y-limits.
for r in range(nrow):
    lo, hi = ylims_per_row[r]
    for ax in axes[r, :]:
        ax.set_ylim(lo, hi)
for ax in axes[-1, :]:
    ax.set_xlabel(r'$1/V$ (V$^{-1}$)')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$\ln(|I|/V^2)$')
fig.tight_layout()
fig.savefig(OUT_DIR / 'FN_field_resolved_grid_all_T_b.png', dpi=200)
plt.show()

## 9. AFM / FM FN endpoint overview vs $T$

$\ln(|I|/V^2)$ vs $1/V$ for the AFM ($|H_\mathrm{z}| < 0.10$ T) and FM (saturated) endpoints, colour-coded by temperature. As $T$ rises the FN slope flattens and the magnetic-state separation collapses, the field-emission fingerprint of the $\sim 80$ K crossover.

In [ ]:
Tnorm = plt.Normalize(vmin=min(T_all), vmax=max(T_all))
cmapT = plt.cm.viridis

fig, (ax_a, ax_f) = plt.subplots(1, 2, figsize=(12, 5), dpi=300, sharey=True)
for T in T_all:
    try:
        df = load_dataframe(DATA_DIR / f'IV_gaussian_{T}K.pkl')
    except FileNotFoundError:
        continue
    V, IA, IF, nA, nF = fn.endpoint_curves(df, T)
    col = cmapT(Tnorm(T))
    for ax, I in [(ax_a, IA), (ax_f, IF)]:
        if I is None:
            continue
        x, yv, _ = fn.fn_transform(V, I)
        ax.plot(x, yv, '-', color=col, lw=1.4)
for ax, lab in [(ax_a, r'AFM ($|H_\mathrm{z}|<0.10$ T)'), (ax_f, 'FM (saturated)')]:
    ax.set_xlabel(r'$1/V$ (V$^{-1}$)')
    ax.set_xlim(0, 20)
    ax.set_ylim(-18, -9)
    ax.text(0.96, 0.04, lab, transform=ax.transAxes, ha='right', va='bottom',
            bbox=dict(facecolor='white', edgecolor='0.7', alpha=0.85,
                      boxstyle='round,pad=0.3'))
ax_a.set_ylabel(r'$\ln(|I|/V^2)$')
sm = plt.cm.ScalarMappable(norm=Tnorm, cmap=cmapT)
sm.set_array([])
cbar = fig.colorbar(sm, ax=[ax_a, ax_f])
cbar.set_label(r'$T$ (K)')
fig.savefig(OUT_DIR / 'FN_endpoint_overview_vs_T.png', dpi=300)
plt.show()


## 6. Save combined CSV

Writes every loaded temperature (not just $T \leq T_{\rm MAX}$) to `T_dependence_summary.csv`.

In [ ]:
out_path = OUT_DIR / 'T_dependence_summary.csv'
summary.to_csv(out_path, index=False)
print(f'Saved {len(summary)} temperatures to {out_path}')
